# Arroz con Leche - Simulacion (interactivo)

Capsula 10. El OTA ya esta dibujado (Bunuelo). Aqui vemos **como se comporta**:
su respuesta en frecuencia (Bode) y los tres numeros que definen si sirve:

- **Av0** = ganancia DC
- **GBW** = producto ganancia-ancho de banda (donde cruza 0 dB)
- **Margen de fase** = estabilidad

Jugamos con los **knobs fisicos** del amplificador (gm, ro, CL, segundo polo) y
vemos moverse el Bode -- la misma intuicion que buscas al simular en ngspice.

> Modelo de **2 polos de primer orden**. Si ya corriste `run_sim.sh ac` en el
> FOSS box, la **ultima seccion** superpone tu Bode **real** de ngspice.

Corre las celdas en orden.

## 1 - El modelo del OTA (2 polos)

Un OTA de 5T, a primer orden, tiene un **polo dominante** en la salida
(`fp1`, por `ro` y la carga `CL`) y un **segundo polo** (`fp2`, el nodo del
espejo). Las relaciones clave:

$$A_{v0} = g_m \cdot r_o \qquad GBW = \frac{g_m}{2\pi C_L} \qquad f_{p1}=\frac{GBW}{A_{v0}}$$

Fijate en algo precioso: **el GBW solo depende de `gm` y `CL`**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox

F = np.logspace(0, 9, 600)   # 1 Hz .. 1 GHz

def bode_modelo(gm_uS, ro_kohm, CL_pF, fp2_MHz):
    gm  = gm_uS*1e-6; ro = ro_kohm*1e3; CL = CL_pF*1e-12; fp2 = fp2_MHz*1e6
    Av0 = gm*ro                              # ganancia DC (lineal)
    GBW = gm/(2*np.pi*CL)                     # producto ganancia-ancho banda
    fp1 = GBW/Av0                             # polo dominante
    H = Av0/((1+1j*F/fp1)*(1+1j*F/fp2))
    gain = 20*np.log10(np.abs(H)); phase = np.angle(H, deg=True)
    return gain, phase, Av0, GBW

def cruce0(freq, gain):
    for i in range(len(gain)-1):
        if gain[i] >= 0 >= gain[i+1]:
            lf = np.log10(freq[i]) + (0-gain[i])*(np.log10(freq[i+1])-np.log10(freq[i]))/(gain[i+1]-gain[i])
            return 10**lf
    return None

def margen_fase(freq, gain, phase):
    fu = cruce0(freq, gain)
    if fu is None: return None, None
    phu = np.interp(np.log10(fu), np.log10(freq), phase)
    pm = (180 + phu) % 360
    if pm > 180: pm -= 360
    return fu, pm

print('modelo listo')

## 2 - Juega con el amplificador

- **gm** (par diferencial): mas gm -> mas ganancia **y** mas GBW.
- **ro** (salida, crece con L): mas ro -> mas ganancia (no cambia el GBW).
- **CL** (carga): mas carga -> menos GBW, pero **mas margen de fase** (compensa).
- **fp2** (segundo polo): si se acerca al GBW, el margen de fase **cae** -> inestable.

In [ ]:
def explora(gm_uS=150.0, ro_kohm=670.0, CL_pF=1.0, fp2_MHz=50.0):
    gain, phase, Av0, GBW = bode_modelo(gm_uS, ro_kohm, CL_pF, fp2_MHz)
    fu, pm = margen_fase(F, gain, phase)
    Av0_db = 20*np.log10(Av0)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True)
    estable = (pm is not None and pm > 45)
    col = '#16a34a' if estable else '#dc2626'
    ax1.semilogx(F, gain, color='#C8742B', lw=2)
    ax1.axhline(0, color='#888', lw=0.8, ls='--')
    if fu: ax1.plot([fu],[0],'o',color=col, ms=8)
    ax1.set_ylabel('Ganancia [dB]'); ax1.grid(True, which='both', alpha=0.3)
    titulo = f'Av0={Av0_db:.1f} dB   GBW={GBW/1e6:.1f} MHz'
    titulo += f'   PM={pm:.0f} deg' if pm is not None else '   (no cruza 0 dB)'
    titulo += '  ->  ' + ('ESTABLE' if estable else 'RIESGO')
    ax1.set_title(titulo, color=col)
    ax2.semilogx(F, phase, color='#2563eb', lw=2)
    if fu: ax2.plot([fu], [np.interp(np.log10(fu),np.log10(F),phase)], 'o', color=col, ms=8)
    ax2.set_ylabel('Fase [deg]'); ax2.set_xlabel('Frecuencia [Hz]')
    ax2.grid(True, which='both', alpha=0.3)
    plt.tight_layout(); plt.show()

interact(explora,
         gm_uS=FloatSlider(value=150, min=50, max=400, step=10, description='gm (uS)'),
         ro_kohm=FloatSlider(value=670, min=100, max=3000, step=50, description='ro (kohm)'),
         CL_pF=FloatSlider(value=1.0, min=0.2, max=5, step=0.1, description='CL (pF)'),
         fp2_MHz=FloatSlider(value=50, min=5, max=200, step=5, description='fp2 (MHz)'));

## 3 - Tres experimentos de un minuto

1. **Sube `CL`** (1 -> 5 pF): el GBW baja, pero el **margen de fase sube**. Esa
   es la idea de **compensar** un amplificador: pagar ancho de banda por estabilidad.
2. **Baja `fp2`** (50 -> 10 MHz): el segundo polo se mete en banda y el **margen
   de fase se desploma** -> el OTA oscila.
3. **Sube `ro`** (L mas largo): la ganancia DC sube, el GBW **no cambia** ->
   confirma que `GBW = gm/(2*pi*CL)` no depende de `ro`.

## 4 - (Opcional) Superpon tu Bode real de ngspice

Si corriste `./run_sim.sh ac` en el FOSS box, sube el `results/bode.txt` a esta
sesion de Colab (icono de carpeta -> subir) y corre la celda: dibuja el Bode
**real** de SKY130 encima del modelo.

In [ ]:
import os
real = None
for cand in ['bode.txt', 'results/bode.txt']:
    if os.path.exists(cand):
        d = np.atleast_2d(np.loadtxt(cand))
        real = (d[:,0], d[:,1], d[:,-1]); break

if real is None:
    print('No encontre bode.txt. Subelo a Colab y vuelve a correr esta celda.')
else:
    fr, gr, pr = real
    o = np.argsort(fr); fr, gr, pr = fr[o], gr[o], pr[o]
    gm_mod, phm_mod, Av0, GBW = bode_modelo(150, 670, 1.0, 50)
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True)
    a1.semilogx(F, gm_mod, '--', color='#C8742B', lw=1.5, label='modelo 2 polos')
    a1.semilogx(fr, gr, color='#16a34a', lw=2, label='ngspice (real)')
    a1.axhline(0, color='#888', lw=0.8, ls='--'); a1.legend()
    a1.set_ylabel('Ganancia [dB]'); a1.grid(True, which='both', alpha=0.3)
    fu = cruce0(fr, gr)
    a1.set_title('OTA 5T - Bode real vs modelo'
                 + (f'   (real: Av0={gr[0]:.1f} dB, GBW={fu/1e6:.1f} MHz)' if fu else ''))
    a2.semilogx(F, phm_mod, '--', color='#2563eb', lw=1.5, label='modelo')
    a2.semilogx(fr, pr, color='#16a34a', lw=2, label='real')
    a2.set_ylabel('Fase [deg]'); a2.set_xlabel('Frecuencia [Hz]')
    a2.grid(True, which='both', alpha=0.3); a2.legend()
    plt.tight_layout(); plt.show()

## 5 - Lo que sigue

El OTA **amplifica, es estable y (en corners) aguanta**. Pero sigue siendo
esquematico -- numeros en la compu. Falta volverlo **silicio**: dibujar las
capas, y hacer que los pares (M3=M4, M5=M6) queden **identicos** de verdad.

🍚 Arroz con Leche &nbsp;->&nbsp; 🍰 Alfajor (Layout)